# NOAA GHCN-D — Relevamiento reproducible de Argentina

**Objetivo:** medir estaciones, disponibilidad de variables y tamaños **reales** de objetos del bucket público `noaa-ghcn-pds` antes de implementar la ingesta. Este notebook **no escribe en AWS**.

Fuente de partida: [documentación GHCN-D en AWS](https://github.com/awslabs/open-data-docs/tree/main/docs/noaa/noaa-ghcn) y [explorador del bucket vinculado](https://noaa-ghcn-pds.s3.amazonaws.com/index.html).

**Dos niveles de medición:** (1) inventario y tamaños S3 reales, sin descargar observaciones; (2) medición exacta de filas/bytes CSV que pasarían los filtros, opcional, leyendo los archivos de estaciones. Los metadatos del inventario indican **primer y último año**, no continuidad diaria.

## 1. Parámetros y dependencias

Cambiar `PAISES`, `FECHA_DESDE`, `FECHA_HASTA` y `VARIABLES` para otros escenarios. El prefijo S3 por estación se valida con el listado real del bucket; no se supone que los archivos estén disponibles sin comprobarlo.

In [2]:
%pip install boto3

  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ------------------------------------ --- 14.9/16.2 MB 73.8 MB/s eta 0:00:01
   ---------------------------------------- 16.2/16.2 MB 63.8 MB/s  0:00:00
Using cached jmespath-1.1.0-py3-none-any.whl (20 kB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)

   ---------------------------------------- 0/6 [urllib3]
  Attempting uninstall: python-dateutil
   ---------------------------------------- 0/6 [urllib3]
    Found existing installation: python-dateutil 3.0.0
   ---------------------------------------- 0/6 [urllib3]
    Uninstalling python-dateutil-3.0.0:
   ---------------------------------------- 0/6 [urllib3]
      Successfully uninstalled python-dateutil-3.0.0
   ---------------------------------------- 0/6 [urllib3]
   ------ ---------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import csv
import gzip
import io
from pathlib import Path

import boto3
import pandas as pd
from botocore import UNSIGNED
from botocore.config import Config
from botocore.exceptions import ClientError

### Parámetros

In [2]:
BUCKET = "noaa-ghcn-pds"
PAISES = ["AR"]
FECHA_DESDE = "2016-01-01"
FECHA_HASTA = "2026-12-31"
VARIABLES = ["PRCP", "TMAX", "TMIN", "TAVG", "SNOW"]
PREFIJO_ESTACIONES = "csv.gz/by_station/"  # verificar con el explorador del bucket https://noaa-ghcn-pds.s3.amazonaws.com/index.html

In [3]:
CACHE = Path("datos_auxiliares_noaa")
CACHE.mkdir(exist_ok=True)

inicio = pd.Timestamp(FECHA_DESDE)
fin = pd.Timestamp(FECHA_HASTA)

assert inicio <= fin
assert PAISES and VARIABLES

s3 = boto3.client("s3", region_name="us-east-1", config=Config(signature_version=UNSIGNED))
print("Configuración:", PAISES, FECHA_DESDE, FECHA_HASTA, VARIABLES)

Configuración: ['AR'] 2016-01-01 2026-12-31 ['PRCP', 'TMAX', 'TMIN', 'TAVG', 'SNOW']


## 2. Descargar únicamente los metadatos del bucket

Se guardan copias locales para evitar repetir las transferencias durante el análisis. Si se requiere actualizar, cambiar `ACTUALIZAR = True`.

In [4]:
ACTUALIZAR = False
archivos = ["ghcnd-stations.txt", "ghcnd-inventory.txt", "ghcnd-countries.txt"]
for nombre in archivos:
    destino = CACHE / nombre
    if ACTUALIZAR or not destino.exists():
        s3.download_file(BUCKET, nombre, str(destino))
    print(f"{nombre}: {destino.stat().st_size:,} bytes locales")

ghcnd-stations.txt: 11,441,526 bytes locales
ghcnd-inventory.txt: 36,400,398 bytes locales
ghcnd-countries.txt: 3,659 bytes locales


## 3. Interpretar estaciones e inventario (campos de ancho fijo)

En el inventario, `FIRSTYEAR` y `LASTYEAR` indican los extremos de disponibilidad del elemento, **no que todos los años o días intermedios tengan datos**.

In [5]:
estaciones = pd.read_fwf(
    CACHE / "ghcnd-stations.txt", header=None,
    colspecs=[(0,11),(12,20),(21,30),(31,37),(38,40),(41,71)],
    names=["ID","LATITUD","LONGITUD","ELEVACION","ESTADO","NOMBRE"],
    dtype={"ID":str,"ESTADO":str,"NOMBRE":str}
)
inventario = pd.read_fwf(
    CACHE / "ghcnd-inventory.txt", header=None,
    colspecs=[(0,11),(12,20),(21,30),(31,35),(36,40),(41,45)],
    names=["ID","LATITUD","LONGITUD","ELEMENT","FIRSTYEAR","LASTYEAR"],
    dtype={"ID":str,"ELEMENT":str}
)

for tabla in (estaciones,inventario):
    tabla["ID"] = tabla["ID"].str.strip()

inventario["ELEMENT"] = inventario["ELEMENT"].str.strip()

sel_estaciones = estaciones[estaciones.ID.str[:2].isin(PAISES)].copy()
sel_inventario = inventario[
    inventario.ID.isin(sel_estaciones.ID) &
    inventario.ELEMENT.isin(VARIABLES) &
    (inventario.FIRSTYEAR <= fin.year) &
    (inventario.LASTYEAR >= inicio.year)
].copy()

print("Estaciones en países elegidos:", sel_estaciones.ID.nunique())
print("Estaciones candidatas (alguna variable y solapamiento temporal):", sel_inventario.ID.nunique())
print("Entradas estación-variable candidatas:", len(sel_inventario))

resumen_variables = (sel_inventario.groupby("ELEMENT").agg(
    estaciones=("ID","nunique"), primer_anio=("FIRSTYEAR","min"), ultimo_anio=("LASTYEAR","max")
).reindex(VARIABLES))

display(resumen_variables)
display(sel_estaciones.head())

Estaciones en países elegidos: 101
Estaciones candidatas (alguna variable y solapamiento temporal): 84
Entradas estación-variable candidatas: 332


,estaciones,primer_anio,ultimo_anio
ELEMENT,,,
PRCP,81.0,1960.0,2025.0
TMAX,83.0,1908.0,2026.0
TMIN,84.0,1908.0,2026.0
TAVG,84.0,1957.0,2026.0
SNOW,NaN,NaN,NaN


,ID,LATITUD,LONGITUD,ELEVACION,ESTADO,NOMBRE
246,AR000000001,-29.60,-58.15,109.0,NaN,BAIBENE
247,AR000000002,-29.82,-57.42,75.0,NaN,BONPLAND
248,AR000000003,-29.98,-58.28,90.0,NaN,CAZADORES CORRENTINOS
249,AR000000004,-30.78,-58.15,71.0,NaN,CHAJAR355
250,AR000000005,-30.03,-57.82,74.0,NaN,COLONIA LIBERTAD


## 4. Cobertura potencial y distribución

Los conteos siguientes se basan en los intervalos publicados en el inventario; no prueban completitud diaria ni disponibilidad de cada mes.

In [6]:
anios = range(inicio.year, fin.year + 1)

filas = []
for anio in anios:
    cobertura = sel_inventario[(sel_inventario.FIRSTYEAR <= anio) & (sel_inventario.LASTYEAR >= anio)]
    for var in VARIABLES:
        filas.append({"anio":anio,"variable":var,"estaciones_potenciales":cobertura.loc[cobertura.ELEMENT.eq(var),"ID"].nunique()})

cobertura_anual = pd.DataFrame(filas).pivot(index="anio",columns="variable",values="estaciones_potenciales")
display(cobertura_anual.reindex(columns=VARIABLES))

matriz_variables = (sel_inventario.assign(disponible=True)
                    .pivot_table(index="ID",columns="ELEMENT",values="disponible",aggfunc="max",fill_value=False))

print("Estaciones potenciales con PRCP:", int(matriz_variables.get("PRCP",pd.Series(dtype=bool)).sum()))
print("Estaciones potenciales con todas las variables solicitadas:", int(matriz_variables.reindex(columns=VARIABLES,fill_value=False).all(axis=1).sum()))

variable,PRCP,TMAX,TMIN,TAVG,SNOW
anio,,,,,
2016,79,82,82,84,0
2017,79,82,84,84,0
2018,79,82,84,84,0
2019,79,82,84,84,0
2020,81,83,84,84,0
2021,81,83,84,84,0
2022,81,83,84,84,0
2023,80,83,84,84,0
2024,77,81,83,83,0


Estaciones potenciales con PRCP: 81
Estaciones potenciales con todas las variables solicitadas: 0


## 5. Medir tamaños de archivos de estaciones en S3 (sin descargarlos)

Se consulta el listado de objetos bajo el prefijo de archivos comprimidos por estación. Cada `Size` es el tamaño real almacenado en S3 (bytes comprimidos), **no** el tamaño tras filtrar fechas/variables. Se informa si faltan archivos esperados.

Si no se hallan objetos, inspeccionar el explorador y ajustar `PREFIJO_ESTACIONES` antes de continuar.

In [7]:
candidatas = set(sel_inventario.ID)
objetos = []
paginador = s3.get_paginator("list_objects_v2")
for pais in PAISES:
    prefijo = PREFIJO_ESTACIONES + pais
    for pagina in paginador.paginate(Bucket=BUCKET, Prefix=prefijo):
        for objeto in pagina.get("Contents", []):
            key = objeto["Key"]
            identificador = Path(key).name.removesuffix(".csv.gz")
            if identificador in candidatas and key.endswith(".csv.gz"):
                objetos.append({"ID":identificador,"key":key,"bytes_comprimidos":objeto["Size"]})
tamanios = pd.DataFrame(objetos,columns=["ID","key","bytes_comprimidos"]).drop_duplicates("key")
print("Archivos encontrados:",len(tamanios),"de",len(candidatas),"estaciones candidatas")
print("Estaciones sin archivo encontrado:",len(candidatas-set(tamanios.ID)))
print("Volumen total real comprimido: %.3f GiB" % (tamanios.bytes_comprimidos.sum()/1024**3))
display(tamanios.sort_values("bytes_comprimidos",ascending=False).head(15))

if tamanios.empty:
    print("ATENCIÓN: revisar PREFIJO_ESTACIONES en el explorador S3; no inferir volumen cero.")

Archivos encontrados: 84 de 84 estaciones candidatas
Estaciones sin archivo encontrado: 0
Volumen total real comprimido: 0.016 GiB


,ID,key,bytes_comprimidos
22,AR000875850,csv.gz/by_station/AR000875850.csv.gz,409148
13,AR000087623,csv.gz/by_station/AR000087623.csv.gz,367744
4,AR000087129,csv.gz/by_station/AR000087129.csv.gz,364938
12,AR000087534,csv.gz/by_station/AR000087534.csv.gz,355915
20,AR000870470,csv.gz/by_station/AR000870470.csv.gz,353375
18,AR000087860,csv.gz/by_station/AR000087860.csv.gz,350431
23,AR000877500,csv.gz/by_station/AR000877500.csv.gz,348704
7,AR000087257,csv.gz/by_station/AR000087257.csv.gz,348468
17,AR000087828,csv.gz/by_station/AR000087828.csv.gz,346364
14,AR000087692,csv.gz/by_station/AR000087692.csv.gz,346348


## 6. Medición exacta del subconjunto CSV (OPCIONAL)

**Este bloque descarga/lee observaciones de las estaciones elegidas**, pero no escribe en AWS. Para mantener control del volumen, está desactivado por defecto. Al activarlo, se recorren los archivos gzip mediante streaming, se aplican fechas/variables y se cuenta el número y tamaño real de filas CSV seleccionadas. El valor medido no depende de extrapolaciones.

**Cuidado:** el formato `by_station` puede tener cabecera CSV; el código detecta y excluye la cabecera del conteo. El tamaño se calcula para las filas seleccionadas como bytes UTF-8 con salto de línea `\n`; no incluye encabezado ni compresión del futuro RAW. Si el formato difiere, se detiene para revisar.

In [9]:
MEDIR_OBSERVACIONES = True  # cambiar a True SOLO después de revisar el volumen del paso 5
MAX_ESTACIONES = None  # None = todas; número entero = prueba limitada, NO volumen total

if MEDIR_OBSERVACIONES:
    if tamanios.empty:
        raise RuntimeError("No se encontraron archivos: verificar prefijo antes de leer")
    resultados=[]
    elegidos=tamanios.sort_values("ID")
    if MAX_ESTACIONES is not None:
        elegidos=elegidos.head(MAX_ESTACIONES)
    desde = inicio.strftime("%Y%m%d")
    hasta = fin.strftime("%Y%m%d")
    for idx, obj in enumerate(elegidos.itertuples(index=False), 1):
        respuesta=s3.get_object(Bucket=BUCKET,Key=obj.key)
        contados=0
        bytes_csv=0
        por_variable={v:0 for v in VARIABLES}
        with gzip.GzipFile(fileobj=respuesta["Body"]) as descomprimido:
            with io.TextIOWrapper(descomprimido,encoding="utf-8",newline="") as entrada:
                for linea in entrada:
                    partes=next(csv.reader([linea]))
                    if partes and partes[0].strip().upper() == "ID":
                        continue
                    if len(partes) < 4:
                        raise ValueError(f"Fila inesperada en {obj.key}: {linea[:150]}")
                    fecha=partes[1].strip().replace("-","")
                    variable=partes[2].strip()
                    if desde <= fecha <= hasta and variable in por_variable:
                        contados+=1
                        por_variable[variable]+=1
                        bytes_csv+=len(linea.rstrip("\r\n").encode("utf-8"))+1
        resultados.append({"ID":obj.ID,"filas_filtradas":contados,"bytes_csv_filtrado":bytes_csv,**por_variable})
        if idx % 25 == 0:
            print(f"Procesadas {idx}/{len(elegidos)} estaciones")
    medicion=pd.DataFrame(resultados)
    display(medicion.head())
    print("Estaciones medidas:",len(medicion),"de",len(tamanios))
    print("Filas realmente seleccionadas:",medicion.filas_filtradas.sum())
    print("Bytes CSV seleccionados (sin cabeceras):",medicion.bytes_csv_filtrado.sum())
    display(medicion[VARIABLES].sum().rename("filas por variable").to_frame())
else:
    print("Medición exacta de observaciones desactivada; no se descargaron archivos por estación.")

Procesadas 25/84 estaciones
Procesadas 50/84 estaciones
Procesadas 75/84 estaciones


,ID,filas_filtradas,bytes_csv_filtrado,PRCP,TMAX,TMIN,TAVG,SNOW
0,AR000000011,8928,313716,1030,1479,3059,3360,0
1,AR000087007,9791,341536,824,2357,3257,3353,0
2,AR000087065,2546,89932,43,557,938,1008,0
3,AR000087078,5018,176796,492,792,1675,2059,0
4,AR000087129,9015,316859,794,1495,3216,3510,0


Estaciones medidas: 84 de 84
Filas realmente seleccionadas: 661335
Bytes CSV seleccionados (sin cabeceras): 23157372


,filas por variable
PRCP,69772
TMAX,122805
TMIN,219587
TAVG,249171
SNOW,0


## 7. Interpretación de resultados

- **Inventario:** cuantifica estaciones candidatas, no observaciones efectivas.
- **Listado S3:** mide el volumen real comprimido de archivos completos por estación. Incluye otros años y variables presentes en ellos.
- **Lectura opcional:** mide filas y bytes CSV seleccionados para las fechas/variables indicadas; permite decidir si el método por estación es conveniente.
- **Limitación:** los cambios diarios en el bucket hacen que los resultados dependan de la fecha de ejecución. Guardar la salida y la fecha para documentar decisiones.